# Lab 04 · Federate MCP tools behind one endpoint

Publish the public Microsoft Learn MCP server through your AI Gateway, add a second backend under another namespace,
and call the federated tools with raw JSON-RPC and the MCP Python SDK. This notebook is the Python track of the
[lab guide](README.md): it deploys the same [infra/tool-server.bicep](infra/tool-server.bicep) as the Bicep + azd track to create the
`learn` tool server, then runs the same checks.

**Get a key → Deploy tool server → Add second backend → Handshake → List tools → Call a tool → Verify**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Tool server `learn` (`toolServers`) | One governed MCP endpoint at `<gateway-url>/default/toolservers/learn/mcp`. |
| Backend `learn` (required) | The Microsoft Learn MCP server, `https://learn.microsoft.com/api/mcp`. Its tools appear as `learn_<tool>`. |
| Backend `docs` (optional) | The same server under a second namespace, so you can see federation without extra credentials. Its tools appear as `docs_<tool>`. |
| Runtime API key `app-first-call` | Authenticates the MCP client. The MCP endpoint accepts it only in the `api-key` header. |
| [infra/tool-server.bicep](infra/tool-server.bicep) | Declares the tool server and its backends. Its `secondNamespace` parameter adds the optional backend. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, keys, and retries. |

![One MCP endpoint on the gateway federates tools from several backends](../assets/diagrams/mcp-federation.svg)

## Prerequisites

- An Azure subscription where you can create resources (Contributor on the resource group) and assign roles
  (Owner, User Access Administrator, or Role Based Access Control Administrator).
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) has run, so the gateway exists.
  Notebook 02 is optional: this notebook reuses its `app-first-call` key, or creates it.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.

In [ ]:
%pip install -q --upgrade azure-identity requests httpx "mcp>=1.30,<2"

### 0. Initialize the lab

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway without edits. The tool server is named
`learn`, like in the lab guide; lab 05 adds tool controls to it.

In [ ]:
import sys
sys.path.insert(0, "../shared")   # helpers that every lab notebook shares
import utils

lab = utils.Lab("04-federate-mcp-tools")
server_name = "learn"   # tool server name, part of the MCP endpoint URL

### 1. Get a runtime API key

MCP clients authenticate to the gateway with the same runtime keys as model clients. `lab.api_key` reuses
`app-first-call` from notebook 02, or creates it, and reports whether it created the key, so clean-up removes only
what this notebook added.

In [ ]:
api_key, created_key = lab.api_key("app-first-call")
print(("✅ Created" if created_key else "♻️ Reusing") + f" key app-first-call: {api_key[:4]}…")

### 2. Deploy the Learn MCP server

MCP servers are `toolServers` child resources of the gateway's `default` workspace.
[infra/tool-server.bicep](infra/tool-server.bicep) registers the Microsoft Learn MCP server as one backend with the namespace
`learn`. The namespace prefixes the backend's tool names, and `required: true` means the server fails if that
backend is down. The template's `mcpEndpoint` output is the governed endpoint that clients call.

> A deployment sets the tool server's properties to what the template declares. Run this notebook before lab 05,
> which adds tool controls to the same server.

In [ ]:
server_path    = f"{lab.workspace_id}/toolServers/{server_name}"
created_server = lab.arm("GET", server_path) is None   # clean-up deletes the server only if this notebook created it

outputs = lab.deploy("infra/tool-server.bicep", gatewayName=lab.gateway_name, serverName=server_name)
mcp_url = outputs["mcpEndpoint"]
print(f"🧰 MCP endpoint: {mcp_url}")
assert mcp_url.endswith(f"/default/toolservers/{server_name}/mcp"), mcp_url

### 3. Federate a second backend

Federation pays off when one endpoint combines several backends. To see it without extra credentials, deploy the
template again with `secondNamespace="docs"`. It adds the same Learn server under the namespace `docs`, marked
**not required**: if an optional backend is unavailable, the gateway skips it instead of failing the whole server.
The redeployment updates the server in place, so the endpoint URL doesn't change.

In [ ]:
lab.deploy("infra/tool-server.bicep", gatewayName=lab.gateway_name, serverName=server_name, secondNamespace="docs")

for e in lab.arm("GET", server_path)["properties"]["endpoints"]:
    print(f"🔌 {e['namespace']:<6} required={e.get('required')!s:<5} {e['mcp']['url']}")

### 4. Run the MCP handshake and list tools

MCP clients start with a handshake: `initialize`, then `notifications/initialized`, then `tools/list`. Streamable HTTP
servers answer with JSON or server-sent events (SSE), and `utils.mcp_session` and `utils.mcp_request` read both.
After `initialize`, the client sends the negotiated `MCP-Protocol-Version` and the `Mcp-Session-Id` the server
returned. This is what [infra/list-tools.sh](infra/list-tools.sh) does.

The new backend can take a minute to show up, so the cell retries until tools from both namespaces appear.

In [ ]:
import time
import requests

for attempt in range(12):
    try:
        headers, server_info = utils.mcp_session(mcp_url, api_key)
        tools = utils.mcp_request(mcp_url, headers, "tools/list")["result"]["tools"]
    except (requests.HTTPError, KeyError) as error:
        print(f"⏳ {error}")
        tools = []
    namespaces = {t["name"].split("_", 1)[0] for t in tools}
    if {"learn", "docs"} <= namespaces:
        break
    print(f"⏳ Waiting for both backends ({sorted(namespaces)})")
    time.sleep(15)

print(f"🤝 Server: {server_info['serverInfo'].get('name')}  protocol: {server_info['protocolVersion']}\n")
for t in tools:
    print(f"🔧 {t['name']:<40} {' '.join((t.get('description') or '').split())[:60]}")
print(f"\n{len(tools)} tools published")
assert {"learn", "docs"} <= namespaces, "Tools from both namespaces should be listed."

### 5. Call a tool through the gateway

Call the Learn search tool by its namespaced name. Tool names are `<namespace>_<tool>`, so the cell picks the name
from the list instead of hard-coding it. The result is Microsoft Learn search results that came through your gateway.

In [ ]:
import json

search_tool = next(t["name"] for t in tools if t["name"].startswith("learn_") and t["name"].endswith("docs_search"))
result = utils.mcp_request(mcp_url, headers, "tools/call", id=3,
                           params={"name": search_tool, "arguments": {"query": "How do I add a token limit in AI Gateway?"}})
text = "\n".join(c.get("text", "") for c in result["result"]["content"] if c.get("type") == "text")
assert not result["result"].get("isError") and text

print(f"📨 {search_tool} returned:")
try:
    hits = json.loads(text)["results"]
    for hit in hits[:3]:
        print(f"📄 {hit['title']}  {hit.get('contentUrl', '')}")
    print(f"   {' '.join(hits[0]['content'].split())[:300]}…")
except (ValueError, KeyError, TypeError, IndexError):
    print(text[:600])

### 6. Connect with the MCP Python SDK

Real clients use an MCP SDK rather than raw JSON-RPC. The SDK runs the same handshake. Pass the runtime key in the
`api-key` header of the HTTP client; this is what [infra/list_tools.py](infra/list_tools.py) does. The cell uses
top-level `await`, which Jupyter supports.

In [ ]:
import httpx
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

async with httpx.AsyncClient(headers={"api-key": api_key}, timeout=60) as http_client:
    async with streamable_http_client(mcp_url, http_client=http_client) as (read, write, _):
        async with ClientSession(read, write) as session:
            init = await session.initialize()
            listed = await session.list_tools()

print(f"🤝 SDK connected to {init.serverInfo.name} (protocol {init.protocolVersion})")
print(f"🔧 {len(listed.tools)} tools: {', '.join(t.name for t in listed.tools)}")
assert len(listed.tools) == len(tools)

### 7. Verify that the gateway checks the caller

Microsoft Learn is public, but your MCP endpoint isn't. Send `tools/list` without a key: the gateway rejects it, and
no backend is called. The MCP endpoint reads the key only from the `api-key` header, so a bearer runtime key is
rejected too.

In [ ]:
probe  = {"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
accept = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}

no_key = requests.post(mcp_url, json=probe, headers=accept, timeout=60)
print(f"📨 Without a key:       HTTP {no_key.status_code}")
assert no_key.status_code == 401

bearer = requests.post(mcp_url, json=probe, headers={**accept, "Authorization": f"Bearer {api_key}"}, timeout=60)
print(f"📨 Key as bearer token: HTTP {bearer.status_code}")
assert bearer.status_code == 401
print("✅ The MCP endpoint only accepts callers with a key in the api-key header")

### 8. Clean up resources

Lab 05 adds tool controls to the `learn` server. If the server existed before this notebook ran, the cell keeps it
and redeploys the template without `secondNamespace`, which removes only the `docs` backend. It deletes the server
and the key only if this notebook created them. Set `keep_for_lab_05 = True` to keep everything.

In [ ]:
keep_for_lab_05 = False   # change to True to keep the tool server for lab 05

if keep_for_lab_05:
    print(f"ℹ️ Kept tool server {server_name} and key app-first-call")
else:
    if created_server:
        lab.delete(server_path)
    else:
        lab.deploy("infra/tool-server.bicep", gatewayName=lab.gateway_name, serverName=server_name)
        print(f"🗑️ Removed backend docs from {server_name}")
    if created_key:
        lab.delete(f"{lab.gateway_id}/apiKeys/app-first-call")

## Troubleshooting

- **`401` from the MCP endpoint with a key**: send the key in the `api-key` header. `Authorization: Bearer <key>`
  works for models but not for MCP. A new key can take a minute to work.
- **Only `learn_` tools are listed**: the `docs` backend isn't applied yet. Wait a minute, rerun step 4, and check
  `lab.arm("GET", server_path)["properties"]["endpoints"]`.
- **JSON-RPC `error` instead of a result**: the server needs the full handshake. Send `initialize` and
  `notifications/initialized` first, and pass `MCP-Protocol-Version` and `Mcp-Session-Id` on later requests.
- **`406 Not Acceptable`**: include `Accept: application/json, text/event-stream`.
- **`Unknown tool`**: tool names carry the namespace prefix, for example `learn_microsoft_docs_search`. Use the names
  from `tools/list`.
- **Tool controls from lab 05 are gone**: a deployment sets the server's properties to what the template declares.
  Redo lab 05 after this notebook.
- **`ImportError: streamable_http_client`**: install `mcp>=1.30,<2`, then restart the kernel.

## Next steps

- Control which tools agents can call in [lab 05](../05-mcp-tool-controls/README.md). Set `keep_for_lab_05 = True` in step 8 first.
- Throttle model callers with [token limits](../06-token-limits/README.md) and its notebook [06-token-limits.ipynb](../06-token-limits/06-token-limits.ipynb).
- Docs: [Connect to MCP tools through AI Gateway](https://ai.gateway.azure.com/docs/mcp-access).
- Read this lab on the labs site: [https://azure.github.io/ai-gateway/labs/04-federate-mcp-tools/](https://azure.github.io/ai-gateway/labs/04-federate-mcp-tools/).